In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#学習データ(train.csv)とテストデータ(test.csv)のデータフレームへの読み込み
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
test_ids = test['Id']  # 提出用に保存

In [ ]:
pip install japanize-matplotlib

In [ ]:
# インポート
# import pandas as pd               # データ操作・分析用（表形式のデータ処理に便利）
import matplotlib.pyplot as plt   # 基本的なグラフ描画ライブラリ（折れ線・棒グラフなど）
import seaborn as sns             # 高機能なグラフ描画ライブラリ（ヒートマップや分布図など）
import math                       # 数学関数ライブラリ（対数・平方根などの計算に使用）
import japanize_matplotlib        # matplotlibで日本語を正しく表示するためのライブラリ
import optuna
# import numpy as np

from sklearn.model_selection import train_test_split  # データの分割（学習用とテスト用）に使用
from sklearn.linear_model import LinearRegression      # 線形回帰モデル（連続値の予測に使用）
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
%matplotlib inline

In [ ]:
# 欠損値がある列とその欠損数を表示（学習データ）
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("【学習データの欠損値】")
print(missing_train)

# 欠損値がある列とその欠損数を表示（テストデータ）
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print("\n【テストデータの欠損値】")
print(missing_test)

In [ ]:
# 数値の特徴量を抽出
numeric_train = train.select_dtypes(include=['number']).columns
numeric_test = test.select_dtypes(include=['number']).columns

# 学習データ：欠損値を各列の平均で補完
for col in numeric_train:
    if train[col].isnull().any():
        train[col].fillna(train[col].mean(), inplace=True)

# テストデータ：欠損値を各列の平均で補完
for col in numeric_test:
    if test[col].isnull().any():
        test[col].fillna(test[col].mean(), inplace=True)

In [ ]:
# 数値でない特徴量を抽出し、欠損値を 'Missing' に置き換え
non_numeric_train = train.select_dtypes(exclude=['number']).columns
train[non_numeric_train] = train[non_numeric_train].fillna('Missing')

# 数値でない特徴量を抽出し、欠損値を 'Missing' に置き換え
non_numeric_test = test.select_dtypes(exclude=['number']).columns
test[non_numeric_test] = test[non_numeric_test].fillna('Missing')

In [ ]:
# 欠損値がある列とその欠損数を表示（学習データ）
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print("【学習データの欠損値】")
print(missing_train)

# 欠損値がある列とその欠損数を表示（テストデータ）
missing_test = test.isnull().sum()
missing_test = missing_test[missing_test > 0].sort_values(ascending=False)
print("\n【テストデータの欠損値】")
print(missing_test)

In [ ]:
MSZoning_map = {
    'A': 0,   # 農業地域（最も低い）
    'I': 1,   # 工業地域
    'C': 2,   # 商業地域
    'RH': 3,  # 高密度住宅地域
    'RM': 4,  # 中密度住宅地域
    'RL': 5,  # 低密度住宅地域
    'RP': 6,  # 公園付き低密度住宅地域
    'FV': 7   # 浮遊住宅地域（最も高い）
}
street_map = {
    'Grvl': 0,  # 砂利道
    'Pave': 1   # 舗装道路
}
alley_map = {
    'NA': 0,    # 裏路地なし
    'Grvl': 1,  # 砂利道
    'Pave': 2   # 舗装
}
lotshape_map = {
    'Reg': 3,   # 整形地（最も高評価）
    'IR1': 2,   # やや不整形
    'IR2': 1,   # 不整形
    'IR3': 0    # 非常に不整形（最も低評価）
}
landcontour_map = {
    'Lvl': 3,   # 平坦（最も高評価）
    'HLS': 2,   # 丘陵（中間）
    'Bnk': 1,   # 傾斜（やや低評価）
    'Low': 0    # 窪地（最も低評価）
}
utilities_map = {
    'ELO': 0,       # 電気のみ（最低評価）
    'NoSeWa': 1,    # 電気・ガスのみ
    'NoSewr': 2,    # 浄化槽（下水なし）
    'AllPub': 3     # 全公共設備（最高評価）
}
lotconfig_map = {
    'Inside': 0,    # 角地でない（標準）
    'FR2': 1,       # 2面接道
    'Corner': 2,    # 角地
    'FR3': 3,       # 3面接道（利便性高）
    'CulDSac': 4    # 袋小路（静かで人気）
}
landslope_map = {
    'Sev': 0,   # 急傾斜（最も低評価）
    'Mod': 1,   # 中程度の傾斜
    'Gtl': 2    # 緩やか（最も高評価）
}
neighborhood_map = {
    'MeadowV': 0, 'IDOTRR': 1, 'BrDale': 2, 'OldTown': 3, 'Edwards': 4,
    'SWISU': 5, 'Blueste': 6, 'NAmes': 7, 'BrkSide': 8, 'Sawyer': 9,
    'NPkVill': 10, 'Mitchel': 11, 'SawyerW': 12, 'NWAmes': 13, 'Gilbert': 14,
    'Blmngtn': 15, 'CollgCr': 16, 'ClearCr': 17, 'Crawfor': 18, 'Somerst': 19,
    'Veenker': 20, 'StoneBr': 21, 'NoRidge': 22, 'NridgHt': 23
}
condition1_map = {
    'RRAn': 0,   # 鉄道に隣接（最も低評価）
    'RRNe': 1,   # 鉄道に近い
    'RRNn': 1,   # 鉄道に近い
    'RRAe': 1,   # 鉄道に近い
    'Artery': 2, # 幹線道路（騒音あり）
    'Feedr': 3,  # フィーダー道路（中間）
    'Norm': 4,   # 通常のアクセス道路（標準）
    'PosA': 5,   # 公園隣接（高評価）
    'PosN': 5    # ポジティブ近隣（高評価）
}
condition2_map = {
    'RRAn': 0,   # 鉄道に隣接（最も低評価）
    'RRAe': 1,   # 鉄道に近い
    'RRNn': 1,   # 鉄道に近い
    'Artery': 2, # 幹線道路（騒音あり）
    'Feedr': 3,  # フィーダー道路（中間）
    'Norm': 4,   # 通常のアクセス道路（標準）
    'PosA': 5,   # 公園隣接（高評価）
    'PosN': 5    # ポジティブ近隣（高評価）
}
bldgtype_map = {
    'Twnhs': 0,     # タウンハウス中央ユニット（最も低評価）
    'TwnhsE': 1,    # タウンハウス端ユニット
    'Duplex': 2,    # 二戸建て住宅
    '2fmCon': 3,    # 二世帯住宅
    '1Fam': 4       # 一戸建て住宅（最も高評価）
}
housestyle_map = {
    '2.5Unf': 0,   # 2.5階建て未完成（最低評価）
    '1.5Unf': 1,   # 1.5階建て未完成
    'SLvl': 2,     # スプリットレベル（複雑構造）
    'Sfoyer': 3,   # スプリット玄関（中間階玄関）
    '1.5Fin': 4,   # 1.5階建て完成
    '1Story': 5,   # 平屋建て（安定人気）
    '2Story': 6,   # 2階建て（高評価）
    '2.5Fin': 7    # 2.5階建て完成（最も高評価）
}
roofstyle_map = {
    'Flat': 0,     # 平らな屋根（排水性や耐久性に課題）
    'Shed': 1,     # 片流れ屋根（モダンだが特殊）
    'Gambrel': 2,  # 納屋風（カントリースタイル）
    'Mansard': 3,  # フランス風（高級感あるが特殊）
    'Gable': 4,    # 三角形の切妻屋根（一般的で人気）
    'Hip': 5       # 寄棟屋根（安定性・耐風性が高く最も高評価）
}
roofmatl_map = {
    'Roll': 0,       # ロール材（最も安価・耐久性低）
    'Tar&Grv': 1,    # タール＆砂利（古い商業用）
    'Membran': 2,    # 防水膜（特殊）
    'WdShngl': 3,    # 木製シングル材（見た目良いが耐久性やや低）
    'WdShake': 4,    # 木製シェイク材（高級感あるがメンテ必要）
    'Metal': 5,      # 金属屋根（耐久性高・モダン）
    'CompShg': 6,    # 合成シングル材（最も一般的・バランス良）
    'ClyTile': 7     # 粘土タイル（高級・耐久性・美観ともに優）
}
exterior_map = {
    'AsphShn': 0,     # アスファルト（最低評価）
    'AsbShng': 1,     # アスベスト（古くて安全性に課題）
    'CBlock': 2,      # コンクリートブロック（無骨・特殊）
    'Other': 3,       # その他（不明・特殊）
    'ImStucc': 4,     # 模造スタッコ（見た目は良いが安価）
    'HdBoard': 5,     # ハードボード（中間）
    'Plywood': 6,     # 合板（中間）
    'Wd Sdng': 7,     # 木製外壁（見た目良いがメンテ必要）
    'Wd Shng': 8,     # 木製シングル（高級感あるが耐久性やや低）
    'Stucco': 9,      # スタッコ（断熱性・美観良好）
    'MetalSd': 10,    # 金属外壁（耐久性高・モダン）
    'Brk Cmn': 11,    # 一般的なレンガ（重厚感あり）
    'BrkFace': 12,    # レンガ調外壁（高級感あり）
    'CmentBd': 13,    # セメント板（耐久性・断熱性良好）
    'VinylSd': 14,    # ビニール外壁（最も人気・メンテ容易）
    'Stone': 15       # 石材（最も高級・耐久性抜群）
}
masvnrtype_map = {
    'NA': 0,        # データなし（最低評価）
    'None': 1,      # 装飾レンガなし
    'BrkCmn': 2,    # 一般的なレンガ（重厚感あり）
    'BrkFace': 3,   # レンガ調の外装（高級感あり）
    'Stone': 4      # 石材（最も高級・耐久性抜群）
}
exterqual_map = {
    'Fa': 0,    # やや劣る（最低評価）
    'TA': 1,    # 平均的（標準）
    'Gd': 2,    # 良好（高評価）
    'Ex': 3     # 最高品質（最も高評価）
}
extercond_map = {
    'Po': 0,    # 劣化が激しい（最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均的（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 最高品質（最も高評価）
}
foundation_map = {
    'Wood': 0,      # 木製基礎（耐久性に劣る・最低評価）
    'Stone': 1,     # 石（古くて特殊）
    'Slab': 2,      # スラブ基礎（安価・中間評価）
    'BrkTil': 3,    # レンガとタイル（見た目良いがやや古い）
    'Cblock': 4,    # コンクリートブロック（一般的・安定）
    'Pconc': 5      # 打設コンクリート（最も高評価・耐久性抜群）
}
bsmtqual_map = {
    'NA': 0,    # データなし（地下室なし、最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均的（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 最高品質（最も高評価）
}
bsmtcond_map = {
    'NA': 0,    # データなし（地下室なし、最低評価）
    'Po': 1,    # 劣化が激しい
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均的（標準）
    'Gd': 4     # 良好（最も高評価）
}
bsmtexposure_map = {
    'NA': 0,    # 地下室なし（最低評価）
    'No': 1,    # 露出なし
    'Mn': 2,    # 最小限の露出
    'Av': 3,    # 平均的な露出
    'Gd': 4     # 良好な露出（最も高評価）
}
bsmtfintype_map = {
    'NA': 0,     # 地下室なし（最低評価）
    'Unf': 1,    # 仕上げなし
    'LwQ': 2,    # 低品質の居住空間
    'BLQ': 3,    # 下位の居住空間
    'Rec': 4,    # レクリエーションルーム（中間評価）
    'ALQ': 5,    # 平均的な居住空間
    'GLQ': 6     # 高品質の仕上げ（最も高評価）
}
heating_map = {
    'Grav': 0,     # 重力式暖房（旧式・最低評価）
    'Wall': 1,     # 壁付け暖房（局所的・やや旧式）
    'OthW': 2,     # その他の温水・蒸気暖房（特殊）
    'GasW': 3,     # ガス温水または蒸気暖房（中間評価）
    'GasA': 4,     # ガス温風暖房（一般的・効率良）
    'Floor': 5     # 床暖房（快適性・効率性ともに最も高評価）
}
heatingqc_map = {
    'Po': 0,    # 劣る（最低評価）
    'Fa': 1,    # やや劣る
    'TA': 2,    # 平均（標準）
    'Gd': 3,    # 良好（高評価）
    'Ex': 4     # 優秀（最も高評価）
}
centralair_map = {
    'N': 0,    # なし（最低評価）
    'Y': 1     # あり（高評価）
}
electrical_map = {
    'NA': 0,      # 電気設備なし（最低評価）
    'FuseP': 1,   # 安全性に懸念あり（旧式）
    'FuseF': 2,   # やや古い設備
    'FuseA': 3,   # 平均的な安全性
    'Mix': 4,     # 複数の配線方式が混在（やや特殊）
    'SBrkr': 5    # 標準ブレーカー（最も一般的・高評価）
}
kitchenqual_map = {
    'Fa': 0,    # やや劣る（最低評価）
    'TA': 1,    # 平均（標準）
    'Gd': 2,    # 良好（高評価）
    'Ex': 3     # 優秀（最も高評価）
}
functional_map = {
    'Sev': 0,     # 深刻な欠陥（最低評価）
    'Maj2': 1,    # 重大な欠陥（大）
    'Maj1': 2,    # 重大な欠陥（小）
    'Mod': 3,     # 中程度の欠陥
    'Min2': 4,    # 軽微な欠陥（やや多い）
    'Min1': 5,    # 軽微な欠陥（少ない）
    'Typ': 6      # 標準的な機能性（最も高評価）
}
fireplacequ_map = {
    'NA': 0,    # 暖炉なし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
garagetype_map = {
    'NA': 0,        # ガレージなし（最低評価）
    'CarPort': 1,   # 簡易型（屋根のみ）
    'Detchd': 2,    # 独立型（利便性やや低）
    'Basment': 3,   # 地下ガレージ（特殊・やや高評価）
    'Attchd': 4,    # 住宅に接続（一般的・高評価）
    'BuiltIn': 5,   # 建物一体型（最も高評価）
    '2Types': 6     # 複数タイプ（柔軟性あり・高評価）
}
garagefinish_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Unf': 1,   # 未仕上げ
    'RFn': 2,   # ラフ仕上げ（中間評価）
    'Fin': 3    # 仕上げ済み（最も高評価）
}
garagequal_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
garagecond_map = {
    'NA': 0,    # ガレージなし（最低評価）
    'Po': 1,    # 劣る
    'Fa': 2,    # やや劣る
    'TA': 3,    # 平均
    'Gd': 4,    # 良好
    'Ex': 5     # 優秀（最も高評価）
}
paveddrive_map = {
    'N': 0,    # 砂利（最低評価）
    'P': 1,    # 部分舗装（中間評価）
    'Y': 2     # 完全舗装（最も高評価）
}
poolqc_map = {
    'NA': 0,    # プールなし（最低評価）
    'Fa': 1,    # やや劣る
    'Gd': 2,    # 良好
    'Ex': 3     # 優秀（最も高評価）
}
fence_map = {
    'NA': 0,       # フェンスなし（最低評価）
    'MnWw': 1,     # 最低限の木製またはワイヤーフェンス
    'MnPrv': 2,    # 最低限のプライバシーフェンス
    'GdWo': 3,     # 高品質な木製フェンス
    'GdPrv': 4     # 高品質なプライバシーフェンス（最も高評価）
}
miscfeature_map = {
    'NA': 0,     # その他設備なし（最低評価）
    'Othr': 1,   # その他（内容不明・低評価）
    'Shed': 2,   # 収納小屋（実用的）
    'Gar2': 3,   # 2つ目のガレージ（利便性・高評価）
    'TenC': 4    # テニスコート（希少・最も高評価）
}
saletype_map = {
    'Oth': 0,      # その他（内容不明・最低評価）
    'COD': 1,      # 裁判所・遺産売買（特殊・不安定）
    'ConLw': 2,    # 頭金・金利ともに低い契約
    'ConLD': 3,    # 頭金が低い契約
    'ConLI': 4,    # 金利が低い契約
    'Con': 5,      # 通常の契約売買
    'CWD': 6,      # 保証付き現金売買
    'WD': 7,       # 保証付き通常売買（安定・高評価）
    'New': 8       # 新築（最も高評価）
}
salecondition_map = {
    'Partial': 0,    # 未完成住宅（価格不安定・最低評価）
    'Abnorml': 1,    # 異常売買（特殊・不安定）
    'AdjLand': 2,    # 隣接地購入（特殊）
    'Alloca': 3,     # 分割売買（やや特殊）
    'Family': 4,     # 家族間売買（価格が市場と乖離しやすい）
    'Normal': 5      # 通常売買（最も安定・高評価）
}

In [ ]:
train['MSZoning'] = train['MSZoning'].map(MSZoning_map).fillna(0)
test['MSZoning'] = test['MSZoning'].map(MSZoning_map).fillna(0)

train['Street'] = train['Street'].map(street_map).fillna(0)
test['Street'] = test['Street'].map(street_map).fillna(0)

train['Alley'] = train['Alley'].map(alley_map).fillna(0)
test['Alley'] = test['Alley'].map(alley_map).fillna(0)

train['LotShape'] = train['LotShape'].map(lotshape_map).fillna(0)
test['LotShape'] = test['LotShape'].map(lotshape_map).fillna(0)

train['LandContour'] = train['LandContour'].map(landcontour_map).fillna(0)
test['LandContour'] = test['LandContour'].map(landcontour_map).fillna(0)

train['Utilities'] = train['Utilities'].map(utilities_map).fillna(0)
test['Utilities'] = test['Utilities'].map(utilities_map).fillna(0)

train['LotConfig'] = train['LotConfig'].map(lotconfig_map).fillna(0)
test['LotConfig'] = test['LotConfig'].map(lotconfig_map).fillna(0)

train['LandSlope'] = train['LandSlope'].map(landslope_map).fillna(0)
test['LandSlope'] = test['LandSlope'].map(landslope_map).fillna(0)

train['Neighborhood'] = train['Neighborhood'].map(neighborhood_map).fillna(0)
test['Neighborhood'] = test['Neighborhood'].map(neighborhood_map).fillna(0)

train['Condition1'] = train['Condition1'].map(condition1_map).fillna(0)
test['Condition1'] = test['Condition1'].map(condition1_map).fillna(0)

train['Condition2'] = train['Condition2'].map(condition2_map).fillna(0)
test['Condition2'] = test['Condition2'].map(condition2_map).fillna(0)

train['BldgType'] = train['BldgType'].map(bldgtype_map).fillna(0)
test['BldgType'] = test['BldgType'].map(bldgtype_map).fillna(0)

train['HouseStyle'] = train['HouseStyle'].map(housestyle_map).fillna(0)
test['HouseStyle'] = test['HouseStyle'].map(housestyle_map).fillna(0)

train['RoofStyle'] = train['RoofStyle'].map(roofstyle_map).fillna(0)
test['RoofStyle'] = test['RoofStyle'].map(roofstyle_map).fillna(0)

train['RoofMatl'] = train['RoofMatl'].map(roofmatl_map).fillna(0)
test['RoofMatl'] = test['RoofMatl'].map(roofmatl_map).fillna(0)

train['Exterior1st'] = train['Exterior1st'].map(exterior_map).fillna(0)
test['Exterior1st'] = test['Exterior1st'].map(exterior_map).fillna(0)

train['Exterior2nd'] = train['Exterior2nd'].map(exterior_map).fillna(0)
test['Exterior2nd'] = test['Exterior2nd'].map(exterior_map).fillna(0)

train['MasVnrType'] = train['MasVnrType'].map(masvnrtype_map).fillna(0)
test['MasVnrType'] = test['MasVnrType'].map(masvnrtype_map).fillna(0)

train['ExterQual'] = train['ExterQual'].map(exterqual_map).fillna(0)
test['ExterQual'] = test['ExterQual'].map(exterqual_map).fillna(0)

train['ExterCond'] = train['ExterCond'].map(extercond_map).fillna(0)
test['ExterCond'] = test['ExterCond'].map(extercond_map).fillna(0)

train['Foundation'] = train['Foundation'].map(foundation_map).fillna(0)
test['Foundation'] = test['Foundation'].map(foundation_map).fillna(0)

train['BsmtQual'] = train['BsmtQual'].map(bsmtqual_map).fillna(0)
test['BsmtQual'] = test['BsmtQual'].map(bsmtqual_map).fillna(0)

train['BsmtCond'] = train['BsmtCond'].map(bsmtcond_map).fillna(0)
test['BsmtCond'] = test['BsmtCond'].map(bsmtcond_map).fillna(0)

train['BsmtExposure'] = train['BsmtExposure'].map(bsmtexposure_map).fillna(0)
test['BsmtExposure'] = test['BsmtExposure'].map(bsmtexposure_map).fillna(0)

train['BsmtFinType1'] = train['BsmtFinType1'].map(bsmtfintype_map).fillna(0)
test['BsmtFinType1'] = test['BsmtFinType1'].map(bsmtfintype_map).fillna(0)

train['BsmtFinType2'] = train['BsmtFinType2'].map(bsmtfintype_map).fillna(0)
test['BsmtFinType2'] = test['BsmtFinType2'].map(bsmtfintype_map).fillna(0)

train['Heating'] = train['Heating'].map(heating_map).fillna(0)
test['Heating'] = test['Heating'].map(heating_map).fillna(0)

train['HeatingQC'] = train['HeatingQC'].map(heatingqc_map).fillna(0)
test['HeatingQC'] = test['HeatingQC'].map(heatingqc_map).fillna(0)

train['CentralAir'] = train['CentralAir'].map(centralair_map).fillna(0)
test['CentralAir'] = test['CentralAir'].map(centralair_map).fillna(0)

train['Electrical'] = train['Electrical'].map(electrical_map).fillna(0)
test['Electrical'] = test['Electrical'].map(electrical_map).fillna(0)

train['KitchenQual'] = train['KitchenQual'].map(kitchenqual_map).fillna(0)
test['KitchenQual'] = test['KitchenQual'].map(kitchenqual_map).fillna(0)

train['Functional'] = train['Functional'].map(functional_map).fillna(0)
test['Functional'] = test['Functional'].map(functional_map).fillna(0)

train['FireplaceQu'] = train['FireplaceQu'].map(fireplacequ_map).fillna(0)
test['FireplaceQu'] = test['FireplaceQu'].map(fireplacequ_map).fillna(0)

train['GarageType'] = train['GarageType'].map(garagetype_map).fillna(0)
test['GarageType'] = test['GarageType'].map(garagetype_map).fillna(0)

train['GarageFinish'] = train['GarageFinish'].map(garagefinish_map).fillna(0)
test['GarageFinish'] = test['GarageFinish'].map(garagefinish_map).fillna(0)

train['GarageQual'] = train['GarageQual'].map(garagequal_map).fillna(0)
test['GarageQual'] = test['GarageQual'].map(garagequal_map).fillna(0)

train['GarageCond'] = train['GarageCond'].map(garagecond_map).fillna(0)
test['GarageCond'] = test['GarageCond'].map(garagecond_map).fillna(0)

train['PavedDrive'] = train['PavedDrive'].map(paveddrive_map).fillna(0)
test['PavedDrive'] = test['PavedDrive'].map(paveddrive_map).fillna(0)

train['PoolQC'] = train['PoolQC'].map(poolqc_map).fillna(0)
test['PoolQC'] = test['PoolQC'].map(poolqc_map).fillna(0)

train['Fence'] = train['Fence'].map(fence_map).fillna(0)
test['Fence'] = test['Fence'].map(fence_map).fillna(0)

train['MiscFeature'] = train['MiscFeature'].map(miscfeature_map).fillna(0)
test['MiscFeature'] = test['MiscFeature'].map(miscfeature_map).fillna(0)

train['SaleType'] = train['SaleType'].map(saletype_map).fillna(0)
test['SaleType'] = test['SaleType'].map(saletype_map).fillna(0)

train['SaleCondition'] = train['SaleCondition'].map(salecondition_map).fillna(0)
test['SaleCondition'] = test['SaleCondition'].map(salecondition_map).fillna(0)

In [ ]:
# SHAPプロットより再分類
def recode_overallqual(val):
    if val <= 4:
        return 1  # 低品質
    elif val <= 6:
        return 2  # 中品質
    else:
        return 3  # 高品質
train['OverallQual_recode'] = train['OverallQual'].apply(recode_overallqual)
test['OverallQual_recode'] = test['OverallQual'].apply(recode_overallqual)

def recode_neighborhood(val):
    if val in ['NoRidge', 'StoneBr']:
        return 3
    elif val in ['CollgCr', 'Gilbert', 'Somerst']:
        return 2
    else:
        return 1
train['Neighborhood_recode'] = train['Neighborhood'].apply(recode_overallqual)
test['Neighborhood_recode'] = test['Neighborhood'].apply(recode_overallqual)

def recode_GarageCars(val):
    if val == 0:
        return 0
    elif val == 1:
        return 1
    else:
        return 2
train['GarageCars_recode'] = train['GarageCars'].apply(recode_overallqual)
test['GarageCars_recode'] = test['GarageCars'].apply(recode_overallqual)

In [ ]:
train['GrLivArea_log'] = np.log1p(train['GrLivArea'])
test['GrLivArea_log'] = np.log1p(test['GrLivArea'])

train['SalePrice_log'] = np.log1p(train['SalePrice'])

In [ ]:
# 特徴量とラベルに分ける
train_features = train.drop(['SalePrice', 'SalePrice_log', 'OverallQual', 'Neighborhood', 'GarageCars', 'GrLivArea'], axis=1).values  # 入力データ（OverallQual除外）
train_target = train['SalePrice_log'].values 

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_features_tr, train_features_va, train_target_tr, train_target_va = train_test_split(train_features, train_target, test_size = 0.2, random_state=0)

In [ ]:
train_features_tr.shape

In [ ]:
train_target_tr.shape

In [ ]:
train_features_va.shape

In [ ]:
train_target_tr.shape

In [ ]:
def objective(trial):
    # ハイパーパラメータの探索空間
    max_depth = trial.suggest_int('max_depth', 5, 20)
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_features = trial.suggest_int('max_features', 5, train_features_tr.shape[1])

    # モデル定義
    lgbm_model = RandomForestRegressor(
        max_depth=max_depth,
        n_estimators=n_estimators,
        max_features=max_features,
        random_state=0,
        n_jobs=-1
    )

    # 交差検証（MSEを最小化）
    score = cross_val_score(lgbm_model, train_features_tr, train_target_tr, cv=5, scoring='neg_mean_squared_error')
    return -score.mean()  # Optunaは最小化するので、MSEをそのまま返す

# 最適化実行
rf_study = optuna.create_study(direction='minimize')
rf_study.optimize(objective, n_trials=10)

# 最適パラメータとスコア表示
print("Best parameters:", rf_study.best_params)
print("Best score (MSE):", rf_study.best_value)
print(f"RMSE: {rf_study.best_value**0.5:.2f}")

In [ ]:
def objective(trial):
    # ハイパーパラメータの探索空間
    max_depth = trial.suggest_int('max_depth', 3, 15)  # LightGBMは浅めの木が多い
    num_leaves = trial.suggest_int('num_leaves', 20, 200)
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)

    # モデル定義
    lgbm_model = LGBMRegressor(
        max_depth=max_depth,
        num_leaves=num_leaves,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        random_state=0,
        n_jobs=-1,
        verbose=-1  # ログを抑制
    )

    # 交差検証（MSEを最小化）
    score = cross_val_score(
        lgbm_model, train_features_tr, train_target_tr,
        cv=5, scoring='neg_mean_squared_error'
    )
    return -score.mean()  # Optunaは最小化するのでMSEを返す

# 最適化実行
lgbm_study = optuna.create_study(direction='minimize')
lgbm_study.optimize(objective, n_trials=30)

# 最適パラメータとスコア表示
print("Best parameters:", lgbm_study.best_params)
print("Best score (MSE):", lgbm_study.best_value)
print(f"RMSE: {lgbm_study.best_value**0.5:.2f}")

In [ ]:
def objective(trial):
    # ハイパーパラメータの探索空間（XGBoost用）
    max_depth = trial.suggest_int('max_depth', 3, 15)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 10)
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    gamma = trial.suggest_float('gamma', 0, 5.0)

    # モデル定義（XGBoost）
    xgb_model = XGBRegressor(
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        random_state=0,
        n_jobs=-1,
        tree_method='hist',  # Kaggle環境で高速化
        verbosity=0
    )

    # 交差検証（MSEを最小化）
    score = cross_val_score(
        xgb_model, train_features_tr, train_target_tr,
        cv=5, scoring='neg_mean_squared_error'
    )
    return -score.mean()

# 最適化実行
xgb_study = optuna.create_study(direction='minimize')
xgb_study.optimize(objective, n_trials=30)

# 最適パラメータとスコア表示
print("Best parameters:", xgb_study.best_params)
print("Best score (MSE):", xgb_study.best_value)
print(f"RMSE: {xgb_study.best_value**0.5:.2f}")

In [ ]:
# 最適パラメータでモデルを再構築
best_rf_model = RandomForestRegressor(
    max_depth=rf_study.best_params['max_depth'],
    n_estimators=rf_study.best_params['n_estimators'],
    max_features=rf_study.best_params['max_features'],
    random_state=0,
    n_jobs=-1
)

In [ ]:
best_xgb_model = XGBRegressor(
    max_depth=xgb_study.best_params['max_depth'],
    n_estimators=xgb_study.best_params['n_estimators'],
    learning_rate=xgb_study.best_params.get('learning_rate', 0.1),
    subsample=xgb_study.best_params.get('subsample', 1.0),
    colsample_bytree=xgb_study.best_params.get('colsample_bytree', 1.0),
    min_child_weight=xgb_study.best_params.get('min_child_weight', 1.0),  # XGBoost用
    gamma=xgb_study.best_params.get('gamma', 0.0),                        # XGBoost用
    random_state=0,
    n_jobs=-1,
    tree_method='hist',  # Kaggle環境で高速化
    verbosity=0
)

In [ ]:
# 最適パラメータでモデルを再構築
best_lgbm_model = LGBMRegressor(
    max_depth=lgbm_study.best_params['max_depth'],
    n_estimators=lgbm_study.best_params['n_estimators'],
    num_leaves=lgbm_study.best_params.get('num_leaves', 31),  # Optunaで探索していれば利用
    learning_rate=lgbm_study.best_params.get('learning_rate', 0.1),
    subsample=lgbm_study.best_params.get('subsample', 1.0),
    colsample_bytree=lgbm_study.best_params.get('colsample_bytree', 1.0),
    random_state=0,
    n_jobs=-1,
    verbose=-1  # ログを抑制
)

In [ ]:
best_rf_model.fit(train_features_tr, train_target_tr)
best_xgb_model.fit(train_features_tr, train_target_tr)
best_lgbm_model.fit(train_features_tr, train_target_tr)

In [ ]:
rf_pred = best_rf_model.predict(train_features_va)
xgb_pred = best_xgb_model.predict(train_features_va)
lgbm_pred = best_lgbm_model.predict(train_features_va)

In [ ]:
# メタモデル用の特徴量を構築
stacked_features_va = np.column_stack([rf_pred, xgb_pred, lgbm_pred])

In [ ]:
# メタモデルを学習
meta_model = LinearRegression()
meta_model.fit(stacked_features_va, train_target_va)

In [ ]:
# # メタモデルを非線形に変更
# meta_model = GradientBoostingRegressor(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=3,
#     random_state=42
# )
# meta_model.fit(stacked_features_va, train_target_va)


In [ ]:
# 特徴量名を定義（共通）
true_feature_names = train.columns.drop([
    'SalePrice', 'SalePrice_log', 'OverallQual', 'Neighborhood', 'GarageCars', 'GrLivArea'
])
train_features_tr_df = pd.DataFrame(train_features_tr, columns=true_feature_names)

# RandomForestの重要度
rf_importances = best_rf_model.feature_importances_
rf_norm = rf_importances / rf_importances.sum()
rf_sorted_idx = np.argsort(rf_norm)[::-1]
rf_cumulative = np.cumsum(rf_norm[rf_sorted_idx])
rf_selected_features = list(train_features_tr_df.columns[rf_sorted_idx][rf_cumulative <= 0.95])

# XGBoostの重要度
xgb_importances = best_xgb_model.feature_importances_
xgb_norm = xgb_importances / xgb_importances.sum()
xgb_sorted_idx = np.argsort(xgb_norm)[::-1]
xgb_cumulative = np.cumsum(xgb_norm[xgb_sorted_idx])
xgb_selected_features = list(train_features_tr_df.columns[xgb_sorted_idx][xgb_cumulative <= 0.95])

# LightGBMの重要度
lgbm_importances = best_lgbm_model.feature_importances_
lgbm_norm = lgbm_importances / lgbm_importances.sum()
lgbm_sorted_idx = np.argsort(lgbm_norm)[::-1]
lgbm_cumulative = np.cumsum(lgbm_norm[lgbm_sorted_idx])
lgbm_selected_features = list(train_features_tr_df.columns[lgbm_sorted_idx][lgbm_cumulative <= 0.95])

In [ ]:
# モデル別特徴量抽出
X_rf = train[rf_selected_features]
X_xgb = train[xgb_selected_features]
X_lgbm = train[lgbm_selected_features]
y = train['SalePrice_log']

# 分割（モデル別）
X_rf_train, X_rf_val, y_train, y_val = train_test_split(X_rf, y, test_size=0.2, random_state=42)
X_xgb_train, X_xgb_val, _, _ = train_test_split(X_xgb, y, test_size=0.2, random_state=42)
X_lgbm_train, X_lgbm_val, _, _ = train_test_split(X_lgbm, y, test_size=0.2, random_state=42)

# 再学習
best_rf_model.fit(X_rf_train, y_train)
best_xgb_model.fit(X_xgb_train, y_train)
best_lgbm_model.fit(X_lgbm_train, y_train)

# 検証予測
rf_pred = best_rf_model.predict(X_rf_val)
xgb_pred = best_xgb_model.predict(X_xgb_val)
lgbm_pred = best_lgbm_model.predict(X_lgbm_val)

# メタモデル学習
stacked_val = np.column_stack([rf_pred, xgb_pred, lgbm_pred])
meta_model = LinearRegression()
meta_model.fit(stacked_val, y_val)
# meta_model = GradientBoostingRegressor(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=3,
#     random_state=42
# )
meta_model.fit(stacked_val, y_val)

In [ ]:
# # モデル別特徴量抽出
# X_rf = train[rf_selected_features]
# X_xgb = train[xgb_selected_features]
# X_lgbm = train[lgbm_selected_features]
# y = train['SalePrice_log']

# # 分割（モデル別）
# X_rf_train, X_rf_val, y_train, y_val = train_test_split(X_rf, y, test_size=0.2, random_state=42)
# X_xgb_train, X_xgb_val, _, _ = train_test_split(X_xgb, y, test_size=0.2, random_state=42)
# X_lgbm_train, X_lgbm_val, _, _ = train_test_split(X_lgbm, y, test_size=0.2, random_state=42)

# # 再学習
# best_rf_model.fit(X_rf_train, y_train)
# best_xgb_model.fit(X_xgb_train, y_train)
# best_lgbm_model.fit(X_lgbm_train, y_train)

# # 検証予測
# rf_pred = best_rf_model.predict(X_rf_val)
# xgb_pred = best_xgb_model.predict(X_xgb_val)
# lgbm_pred = best_lgbm_model.predict(X_lgbm_val)

# # メタモデル用特徴量構築
# stacked_val = np.column_stack([rf_pred, xgb_pred, lgbm_pred])

# # ✅ メタモデルを非線形に変更（GradientBoosting）
# meta_model = GradientBoostingRegressor(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=3,
#     random_state=42
# )
# meta_model.fit(stacked_val, y_val)


In [ ]:
X_rf_test = test[rf_selected_features]
X_xgb_test = test[xgb_selected_features]
X_lgbm_test = test[lgbm_selected_features]

rf_test_pred = best_rf_model.predict(X_rf_test)
xgb_test_pred = best_xgb_model.predict(X_xgb_test)
lgbm_test_pred = best_lgbm_model.predict(X_lgbm_test)

stacked_test = np.column_stack([rf_test_pred, xgb_test_pred, lgbm_test_pred])
final_pred_log = meta_model.predict(stacked_test)
final_pred = np.expm1(final_pred_log)

In [ ]:
# メタモデルの検証予測
meta_val_pred = meta_model.predict(stacked_val)

# RMSE（対数スケール）
rmse_log = mean_squared_error(y_val, meta_val_pred, squared=False)
print(f"Meta model RMSE (log scale): {rmse_log:.4f}")

# R²（対数スケール）
r2_log = r2_score(y_val, meta_val_pred)
print(f"Meta model R² (log scale): {r2_log:.4f}")

# 元スケールに戻す（任意）
y_val_exp = np.expm1(y_val)
meta_val_pred_exp = np.expm1(meta_val_pred)

# RMSE（元スケール）
rmse_exp = mean_squared_error(y_val_exp, meta_val_pred_exp, squared=False)
print(f"Meta model RMSE (original scale): {rmse_exp:.2f}")

# R²（元スケール）
r2_exp = r2_score(y_val_exp, meta_val_pred_exp)
print(f"Meta model R² (original scale): {r2_exp:.4f}")

In [ ]:
# 特徴量名を定義
true_feature_names = train.columns.drop([
    'SalePrice', 'SalePrice_log', 'OverallQual', 'Neighborhood', 'GarageCars', 'GrLivArea'
])

# DataFrameに変換（列名付き）
train_features_tr_df = pd.DataFrame(train_features_tr, columns=true_feature_names)

# 重要度取得と累積寄与率で特徴量選定 
importances = best_lgbm_model.feature_importances_
# importances = best_xgb_model.feature_importances_
importances_norm = importances / importances.sum()
sorted_idx = np.argsort(importances_norm)[::-1]
sorted_features = train_features_tr_df.columns[sorted_idx]
cumulative = np.cumsum(importances_norm[sorted_idx])
selected_features = list(sorted_features[cumulative <= 0.95])

In [ ]:
# 特徴量を統一（XGBoostの重要度から選定済みと仮定）
X = train[selected_features]
y = train['SalePrice_log']  # 対数変換済みの目的変数

# 学習・検証に分割（80%学習、20%検証）
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# モデル再学習
best_rf_model.fit(X_train, y_train)
best_xgb_model.fit(X_train, y_train)
best_lgbm_model.fit(X_train, y_train)

# 検証予測
rf_pred = best_rf_model.predict(X_val)
xgb_pred = best_xgb_model.predict(X_val)
lgbm_pred = best_lgbm_model.predict(X_val)

# メタモデル学習
stacked_val = np.column_stack([rf_pred, xgb_pred, lgbm_pred])
meta_model = LinearRegression()
meta_model.fit(stacked_val, y_val)

# テスト予測
X_test = test[selected_features]
rf_test_pred = best_rf_model.predict(X_test)
xgb_test_pred = best_xgb_model.predict(X_test)
lgbm_test_pred = best_lgbm_model.predict(X_test)

stacked_test = np.column_stack([rf_test_pred, xgb_test_pred, lgbm_test_pred])
final_pred_log = meta_model.predict(stacked_test)
final_pred = np.expm1(final_pred_log)  # 対数スケール → 元スケール


In [ ]:
# 提出ファイル作成
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_pred.flatten()
})
submission.to_csv('stacking_submission.csv', index=False)
